# Lab 5.7 &mdash; A Rule-Based ReAct Agent (two-step task)

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Wire tools + memory + a loop + a policy into one working agent
- Solve a TWO-step task: look up the population, then double it
- Confirm both tools were used and the loop terminated

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 5 slides &mdash; The agent loop](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-05-07"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Now we assemble the pieces from Labs 2&ndash;6 into a real **ReAct agent**. The task &mdash;
*"twice the population of Metropolis"* &mdash; needs **two** tools chained: first **lookup** the
population, then **calculator** to double it. The rule-based **policy** reads the memory to decide
what is still missing and what to do next.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)
KNOWLEDGE = {"population of metropolis": "120000", "capital of france": "Paris"}
TOOLS = {
    "lookup":     {"fn": lambda k: KNOWLEDGE.get(k.lower().strip(), "unknown")},
    "calculator": {"fn": safe_calc},
}
print("tools ready:", list(TOOLS))

## Your Turn
Fill the policy's two-step reasoning. `run_agent` (provided) runs the loop and memory for you.

In [ ]:
def policy(goal, memory):
    seen = [s["action"] for s in memory]
    if "lookup" not in seen:
        return ("lookup", ___)   # TODO: the key to look up the population
    pop = int([s["observation"] for s in memory if s["action"] == "lookup"][0])
    if "calculator" not in seen:
        return ("calculator", ___)   # TODO: an expression for twice the population
    answer = [s["observation"] for s in memory if s["action"] == "calculator"][0]
    return (___, str(answer))   # TODO: the action name that ENDS the loop

def run_agent(goal, max_steps=8):
    memory = []
    for _ in range(max_steps):
        action, arg = policy(goal, memory)
        if action == 'final':
            return {'answer': arg, 'memory': memory}
        obs = TOOLS[action]['fn'](arg)
        memory.append({'action': action, 'observation': obs})
    return {'answer': None, 'memory': memory}

try:
    out = run_agent('twice the population of metropolis')
    print('answer:', out['answer'])
    print('actions used:', [s['action'] for s in out['memory']])
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("final answer is twice the population", lambda: run_agent("twice the population of metropolis")["answer"] == "240000")
expect_true("the lookup tool was used", lambda: any(s["action"] == "lookup" for s in run_agent("x")["memory"]))
expect_true("the calculator tool was used", lambda: any(s["action"] == "calculator" for s in run_agent("x")["memory"]))
expect_true("tools ran in order: lookup then calculator", lambda: [s["action"] for s in run_agent("x")["memory"]] == ["lookup", "calculator"])
expect_true("the loop terminated with an answer", lambda: run_agent("x")["answer"] is not None)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Tools + memory + loop + policy = a working ReAct agent that chains two tools. Swap the rule-based policy for an LLM and you have the real thing (Module 6).

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>